In [33]:
import random
from pathlib import Path

import sqlite3

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score
from scipy import stats

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "fifa-2026"
DB_PATH = PROJECT_ROOT / "db" / "fifa2026.db"
DB_PATH.parent.mkdir(exist_ok=True)

PERMA_SEED = 42
random.seed(PERMA_SEED)
np.random.seed(PERMA_SEED)

In [34]:
def q(sql, params=None):
    with sqlite3.connect(DB_PATH) as c:
        return pd.read_sql_query(sql, c, params=params)

In [35]:
final_dataset = q("""
WITH
squad_totals AS (
    SELECT
        team_id,
        AVG((julianday('2026-06-11') - julianday(date_of_birth)) / 365.25) AS squad_age,
        SUM(caps) AS total_caps,
        SUM(goals) AS goals_sum
    FROM squads_and_players
    GROUP BY team_id
),
match_multiplier AS (
    SELECT 
        match_id,
        date, 
        home_team_id AS team_id,
        home_score AS goals_for, 
        away_score AS goals_against
    FROM matches
    UNION ALL
    SELECT 
        match_id, 
        date, 
        away_team_id,
        away_score,
        home_score
    FROM matches
),
team_timeline AS (
    SELECT
        match_id,
        team_id,
        julianday(date) - julianday(LAG(date) OVER w) AS rest_days,
        AVG(goals_for - goals_against) OVER (
            PARTITION BY team_id
            ORDER BY date, match_id
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS recent_form
    FROM match_multiplier
    WINDOW w AS (PARTITION BY team_id ORDER BY date, match_id)
)
SELECT
    matches.match_id,
    matches.home_team_id,
    matches.away_team_id,
    t1.elo_rating - t2.elo_rating AS elo_rating_difference,
    home.squad_age - away.squad_age AS squad_age_difference,
    home.total_caps - away.total_caps AS caps_difference,
    home.goals_sum - away.goals_sum AS goal_total_difference,
    COALESCE(tl_home.rest_days, 10) - COALESCE(tl_away.rest_days, 10) AS rest_days_difference,
    stages.is_knockout AS is_knockout,
    (t1.fifa_code = venues.country) - (t2.fifa_code = venues.country) AS host_advantage,
    COALESCE(tl_home.recent_form, 0) - COALESCE(tl_away.recent_form, 0) AS form_difference,
    matches.home_score - matches.away_score AS goal_difference
FROM matches
JOIN teams AS t1 ON t1.team_id = matches.home_team_id
JOIN teams AS t2 ON t2.team_id = matches.away_team_id
JOIN venues ON venues.venue_id = matches.venue_id
JOIN tournament_stages AS stages ON stages.stage_id = matches.stage_id
JOIN squad_totals AS home ON home.team_id = matches.home_team_id
JOIN squad_totals AS away ON away.team_id = matches.away_team_id
JOIN team_timeline AS tl_home ON tl_home.match_id = matches.match_id AND tl_home.team_id = matches.home_team_id
JOIN team_timeline AS tl_away ON tl_away.match_id = matches.match_id AND tl_away.team_id = matches.away_team_id
ORDER BY matches.match_id;
""")

In [36]:
>>> final_dataset.head()

,match_id,home_team_id,away_team_id,elo_rating_difference,squad_age_difference,caps_difference,goal_total_difference,rest_days_difference,is_knockout,host_advantage,form_difference,goal_difference
0,1,1,2,190,1.092297,699,54,0.0,0,1,0.0,2
1,2,3,4,60,0.444901,236,45,0.0,0,0,0.0,1
2,3,5,6,150,0.516190,360,40,0.0,0,1,0.0,0
3,4,13,14,125,-2.097615,272,74,0.0,0,1,0.0,3
4,5,7,8,-260,1.200653,438,91,0.0,0,0,0.0,0


In [37]:
>>> final_dataset.shape

(104, 12)

In [38]:
ID_COLS = ["match_id", "home_team_id", "away_team_id"]
TARGET = "goal_difference"

FEATURES = [clmn for clmn in final_dataset.columns if clmn not in ID_COLS + [TARGET]]

In [39]:
>>> FEATURES

['elo_rating_difference',
 'squad_age_difference',
 'caps_difference',
 'goal_total_difference',
 'rest_days_difference',
 'is_knockout',
 'host_advantage',
 'form_difference']

In [40]:
# works great! linear reg now

In [41]:
X_train, X_test, y_train, y_test = train_test_split(
    final_dataset[FEATURES], final_dataset[TARGET], test_size=0.2, random_state=PERMA_SEED
)

In [42]:
>>> X_train

,elo_rating_difference,squad_age_difference,caps_difference,goal_total_difference,rest_days_difference,is_knockout,host_advantage,form_difference
55,-65,1.643132,-46,-21,0.0,0,0,-1.000000
22,10,-3.451377,-971,-58,0.0,0,0,0.000000
76,-25,-0.955194,-95,-41,0.0,1,0,0.333333
44,360,-0.177960,204,140,0.0,0,0,2.000000
72,-175,-0.164271,-435,-68,0.0,1,0,-2.000000
...,...,...,...,...,...,...,...,...
71,250,1.353551,591,86,0.0,0,0,-1.000000
14,-300,-0.226189,0,22,0.0,0,0,0.000000
92,-110,1.287843,418,143,0.0,1,0,-0.666667
51,45,-3.051229,-973,-122,0.0,0,0,1.500000


In [43]:
model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)

In [45]:
r2 = r2_score(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
print(f"Test R^2:  {r2:.3f}")
print(f"Test RMSE: {rmse:.3f} goals")

pd.Series(model.coef_, index=FEATURES, name="coefficient").sort_values()

Test R^2:  0.386
Test RMSE: 1.763 goals


is_knockout             -0.778325
form_difference         -0.020167
caps_difference         -0.001280
goal_total_difference    0.001103
elo_rating_difference    0.006445
squad_age_difference     0.084562
rest_days_difference     0.179468
host_advantage           1.168129
Name: coefficient, dtype: float64

In [ ]:
# DONE!

# SECOND QUESTION

In [46]:
goals_pred_ds = q("""
WITH
squad_totals AS (
    SELECT
        team_id,
        AVG((julianday('2026-06-11') - julianday(date_of_birth)) / 365.25) AS squad_age,
        SUM(goals) AS career_goals
        FROM squads_and_players
        GROUP BY team_id
),
team_match AS (
    SELECT match_id,
           date,
           home_team_id AS team_id,
           away_team_id AS opponent_id,
           home_score AS goals_scored,
           away_score AS goals_conceded
    FROM matches
    UNION ALL
    SELECT
        match_id,
        date,
        away_team_id,
        home_team_id,
        away_score,
        home_score
    FROM matches
),
team_timeline AS (
    SELECT
        match_id,
        team_id,
        opponent_id,
        goals_scored,
        julianday(date) - julianday(LAG (date) OVER w) AS rest_days,
        AVG(goals_scored) OVER (
            PARTITION BY team_id
            ORDER BY date, match_id
            ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING
        ) AS recent_goals_scored
    FROM team_match
    WINDOW w AS (PARTITION BY team_id ORDER BY date, match_id)
)
SELECT
    tl.match_id,
    tl.team_id,
    tl.opponent_id,
    t.elo_rating AS team_elo,
    o.elo_rating AS opponent_elo,
    sq.squad_age AS team_squad_age,
    sq.career_goals AS team_career_goals,
    COALESCE(tl.rest_days, 10) AS team_rest_days,
    stages.is_knockout AS is_knockout,
    (t.fifa_code = venues.country) AS team_is_host,
    COALESCE(tl.recent_goals_scored, 0) AS team_recent_goals,
    tl.goals_scored AS goals_scored
FROM team_timeline AS tl
JOIN matches AS m ON m.match_id = tl.match_id
JOIN teams AS t ON t.team_id = tl.team_id
JOIN teams AS o ON o.team_id = tl.opponent_id
JOIN venues ON venues.venue_id = m.venue_id
JOIN tournament_stages AS stages ON stages.stage_id = m.stage_id
JOIN squad_totals AS sq ON sq.team_id = tl.team_id
ORDER BY tl.match_id, tl.team_id;
""")

In [47]:
>>> goals_pred_ds.head()

,match_id,team_id,opponent_id,team_elo,opponent_elo,team_squad_age,team_career_goals,team_rest_days,is_knockout,team_is_host,team_recent_goals,goals_scored
0,1,1,2,1810,1620,27.900700,120,10.0,0,1,0.0,2
1,1,2,1,1620,1810,26.808403,66,10.0,0,0,0.0,0
2,2,3,4,1800,1740,28.065708,144,10.0,0,0,0.0,2
3,2,4,3,1740,1800,27.620808,99,10.0,0,0,0.0,1
4,3,5,6,1795,1645,26.972674,134,10.0,0,1,0.0,1
